In [1]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [optuna]


In [3]:
# import necessary libraries

import optuna
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import pandas as pd

url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv'
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insuline', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']

df = pd.read_csv(url, names=columns)

df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insuline,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [4]:
import numpy as np

cols_with_missing_vals = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insuline', 'BMI']

df[cols_with_missing_vals] = df[cols_with_missing_vals].replace(0, np.nan)

df.fillna(df.mean(), inplace=True)

print(df.isnull().sum())

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insuline                    0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [6]:
x = df.drop('Outcome', axis=1)
y = df['Outcome']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=42)

scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

print(x_train.shape)
print(x_test.shape)



(537, 8)
(231, 8)


# Optuna

In [25]:
from sklearn.model_selection import cross_val_score

def objective(trial):
  #suggest hyper parameter values
  n_estimators = trial.suggest_int('n_estimators', 50, 200)
  max_depth = trial.suggest_int('max_depth', 3, 20)

  # Create the randomforestclassifier with suggested hyper parameter
  model = RandomForestClassifier(
      n_estimators = n_estimators,
      max_depth = max_depth,
      random_state = 42
  )

  # 3-fold cross-validation and calculate accuracy
  accuracy = cross_val_score(model, x_train, y_train, cv=3, scoring='accuracy').mean()

  return accuracy


In [26]:
from sklearn.ensemble import RandomForestClassifier

# create a study object and optimize the objective function
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler())
study.optimize(objective, n_trials=50)

[I 2026-11-07 11:46:09,056] A new study created in memory with name: no-name-408eeb6e-4b73-47d2-9719-f5b51c491ab4
[I 2026-11-07 11:46:09,814] Trial 0 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 135, 'max_depth': 8}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-11-07 11:46:10,241] Trial 1 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 64, 'max_depth': 17}. Best is trial 1 with value: 0.7728119180633147.
[I 2026-11-07 11:46:11,507] Trial 2 finished with value: 0.7541899441340782 and parameters: {'n_estimators': 161, 'max_depth': 3}. Best is trial 1 with value: 0.7728119180633147.
[I 2026-11-07 11:46:12,789] Trial 3 finished with value: 0.7597765363128491 and parameters: {'n_estimators': 148, 'max_depth': 6}. Best is trial 1 with value: 0.7728119180633147.
[I 2026-11-07 11:46:14,121] Trial 4 finished with value: 0.7709497206703911 and parameters: {'n_estimators': 177, 'max_depth': 20}. Best is trial 1 with value: 0.77281191

In [27]:
# print the best resulf
print('Best trial: score {}, params {}'.format(study.best_trial.value, study.best_trial.params))

Best trial: score 0.7783985102420857, params {'n_estimators': 125, 'max_depth': 14}


In [28]:
from sklearn.metrics import accuracy_score

best_model = RandomForestClassifier(
    n_estimators = study.best_trial.params['n_estimators'],
    max_depth = study.best_trial.params['max_depth'],
    random_state = 42
)

best_model.fit(x_train, y_train)

y_pred = best_model.predict(x_test)

accuracy = accuracy_score(y_test, y_pred)

print('Accuracy: {}'.format(accuracy))

Accuracy: 0.7402597402597403


# Optuna visualization

In [29]:
# for visualisation
from optuna.visualization import plot_optimization_history
from optuna.visualization import plot_param_importances
from optuna.visualization import plot_parallel_coordinate
from optuna.visualization import plot_slice

In [30]:
# optimization history
plot_optimization_history(study)

In [31]:
# parallel coordinate plot
plot_parallel_coordinate(study)

In [32]:
# Hyperparameter importance
plot_param_importances(study)

# Random SearchCV

In [14]:
from sklearn.model_selection import cross_val_score

def objective(trial):
  #suggest hyper parameter values
  n_estimators = trial.suggest_int('n_estimators', 50, 200)
  max_depth = trial.suggest_int('max_depth', 3, 20)

  # Create the randomforestclassifier with suggested hyper parameter
  model = RandomForestClassifier(
      n_estimators = n_estimators,
      max_depth = max_depth,
      random_state = 42
  )

  # 3-fold cross-validation and calculate accuracy
  accuracy = cross_val_score(model, x_train, y_train, cv=3, scoring='accuracy').mean()

  return accuracy

In [16]:
from sklearn.ensemble import RandomForestClassifier

# create a study object and optimize the objective function
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.RandomSampler())
study.optimize(objective, n_trials=50)

[I 2026-11-07 11:30:44,790] A new study created in memory with name: no-name-b7a240c5-02a7-4ac8-9cf9-0ab08fb485cf
[I 2026-11-07 11:30:46,850] Trial 0 finished with value: 0.7635009310986964 and parameters: {'n_estimators': 138, 'max_depth': 9}. Best is trial 0 with value: 0.7635009310986964.
[I 2026-11-07 11:30:48,370] Trial 1 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 167, 'max_depth': 12}. Best is trial 1 with value: 0.7728119180633147.
[I 2026-11-07 11:30:50,015] Trial 2 finished with value: 0.7635009310986964 and parameters: {'n_estimators': 198, 'max_depth': 4}. Best is trial 1 with value: 0.7728119180633147.
[I 2026-11-07 11:30:50,938] Trial 3 finished with value: 0.7765363128491621 and parameters: {'n_estimators': 130, 'max_depth': 20}. Best is trial 3 with value: 0.7765363128491621.
[I 2026-11-07 11:30:51,912] Trial 4 finished with value: 0.7635009310986964 and parameters: {'n_estimators': 183, 'max_depth': 5}. Best is trial 3 with value: 0.7765363

In [17]:
# print the best resulf
print('Best trial: score {}, params {}'.format(study.best_trial.value, study.best_trial.params))

Best trial: score 0.7802607076350093, params {'n_estimators': 133, 'max_depth': 7}


In [18]:
from sklearn.metrics import accuracy_score

best_model = RandomForestClassifier(
    n_estimators = study.best_trial.params['n_estimators'],
    max_depth = study.best_trial.params['max_depth'],
    random_state = 42
)

best_model.fit(x_train, y_train)

y_pred = best_model.predict(x_test)

accuracy = accuracy_score(y_test, y_pred)

print('Accuracy: {}'.format(accuracy))

Accuracy: 0.7489177489177489


# Grid Search

In [20]:
search_space = {
    'n_estimators': [50, 100, 150, 200],
    'max_depth': [3, 5, 7, 10]
}

In [21]:
# create a study object and optimize the objective function
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.GridSampler(search_space))
study.optimize(objective, n_trials=50)

[I 2026-11-07 11:40:18,502] A new study created in memory with name: no-name-21fd541e-3565-4c3f-a633-166084d255ee
[I 2026-11-07 11:40:19,019] Trial 0 finished with value: 0.7560521415270017 and parameters: {'n_estimators': 100, 'max_depth': 3}. Best is trial 0 with value: 0.7560521415270017.
[I 2026-11-07 11:40:19,803] Trial 1 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 150, 'max_depth': 5}. Best is trial 1 with value: 0.7653631284916201.
[I 2026-11-07 11:40:20,108] Trial 2 finished with value: 0.7802607076350093 and parameters: {'n_estimators': 50, 'max_depth': 7}. Best is trial 2 with value: 0.7802607076350093.
[I 2026-11-07 11:40:20,945] Trial 3 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 100, 'max_depth': 7}. Best is trial 2 with value: 0.7802607076350093.
[I 2026-11-07 11:40:21,838] Trial 4 finished with value: 0.7616387337057727 and parameters: {'n_estimators': 100, 'max_depth': 10}. Best is trial 2 with value: 0.780260707

In [23]:
# best result and parameter

print('Best trial: score {}, params {}'.format(study.best_trial.value, study.best_trial.params))

Best trial: score 0.7802607076350093, params {'n_estimators': 50, 'max_depth': 7}


In [24]:
# best result and parameter
best_model = RandomForestClassifier(
    n_estimators = study.best_trial.params['n_estimators'],
    max_depth = study.best_trial.params['max_depth'],
    random_state = 42
)

best_model.fit(x_train, y_train)

y_pred = best_model.predict(x_test)

accuracy = accuracy_score(y_test, y_pred)

print('Accuracy: {}'.format(accuracy))


Accuracy: 0.7316017316017316


# Dynamic Search

In [40]:
# define the objective function for Optuna

def objective(trial):
  # choose the name of algorithm
  classifier_name = trial.suggest_categorical('classifier', ['SVM', 'RandomForest', 'GradientBoosting'])

  if classifier_name == 'SVM':
    # suggest hyperparameters for SVM
    C = trial.suggest_float('C', 1e-3, 1e3, log=True)
    kernel = trial.suggest_categorical('kernel', ['linear', 'rbf'])
    gamma = trial.suggest_categorical('gamma', ['scale', 'auto'])

    # create SVM model with suggested hyperparameters
    model = SVC(C=C, kernel=kernel, gamma=gamma)

  if classifier_name == 'RandomForest':
    # suggest hyperparameters for RandomForest
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    max_depth = trial.suggest_int('max_depth', 3, 20)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
    bootstrap = trial.suggest_categorical('bootstrap', [True, False])

    # create RandomForest model with suggested hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        bootstrap=bootstrap
    )

  elif classifier_name == 'GradientBoosting':
    # suggest hyperparameters for GradientBoosting
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    learning_rate = trial.suggest_float('learning_rate', 0.01, 1.0)
    max_depth = trial.suggest_int('max_depth', 3, 10)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)

    model = GradientBoostingClassifier(
        n_estimators=n_estimators,
        learning_rate=learning_rate,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        random_state=42
    )

  # 3-fold cross-validation and calculate accuracy
  accuracy = cross_val_score(model, x_train, y_train, cv=3, scoring='accuracy').mean()

  return accuracy



In [41]:
from sklearn.svm import SVC
from sklearn.ensemble import GradientBoostingClassifier

# Create a study and optimize
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=200)

[I 2026-11-07 14:35:25,198] A new study created in memory with name: no-name-a665d32b-279b-4707-b2e2-4f7352ca3dc1
[I 2026-11-07 14:35:25,233] Trial 0 finished with value: 0.7728119180633147 and parameters: {'classifier': 'SVM', 'C': 1.975558978269067, 'kernel': 'rbf', 'gamma': 'auto'}. Best is trial 0 with value: 0.7728119180633147.
[I 2026-11-07 14:35:26,269] Trial 1 finished with value: 0.7318435754189944 and parameters: {'classifier': 'GradientBoosting', 'n_estimators': 157, 'learning_rate': 0.6444598837549397, 'max_depth': 10, 'min_samples_split': 6, 'min_samples_leaf': 9}. Best is trial 0 with value: 0.7728119180633147.
[I 2026-11-07 14:35:26,506] Trial 2 finished with value: 0.7858472998137801 and parameters: {'classifier': 'SVM', 'C': 60.082922507186645, 'kernel': 'linear', 'gamma': 'auto'}. Best is trial 2 with value: 0.7858472998137801.
[I 2026-11-07 14:35:28,375] Trial 3 finished with value: 0.750465549348231 and parameters: {'classifier': 'GradientBoosting', 'n_estimators': 

In [43]:
print('Best trial: score {}, params {}'.format(study.best_trial.value, study.best_trial.params))

Best trial: score 0.7895716945996275, params {'classifier': 'SVM', 'C': 0.12567799770590302, 'kernel': 'linear', 'gamma': 'scale'}


In [44]:
# create the best model
best_model = SVC(
    C=study.best_trial.params['C'],
    kernel=study.best_trial.params['kernel'],
    gamma=study.best_trial.params['gamma']
)

best_model.fit(x_train, y_train)

y_pred = best_model.predict(x_test)

accuracy = accuracy_score(y_test, y_pred)

print('Accuracy: {}'.format(accuracy))

Accuracy: 0.7359307359307359


In [45]:
study.trials_dataframe()

,number,value,datetime_start,datetime_complete,duration,params_C,params_bootstrap,params_classifier,params_gamma,params_kernel,params_learning_rate,params_max_depth,params_min_samples_leaf,params_min_samples_split,params_n_estimators,system_attrs_tpe:relative_params:0,state
0,0,0.772812,2026-11-07 14:35:25.201044,2026-11-07 14:35:25.232920,0 days 00:00:00.031876,1.975559,NaN,SVM,auto,rbf,NaN,NaN,NaN,NaN,NaN,NaN,COMPLETE
1,1,0.731844,2026-11-07 14:35:25.234006,2026-11-07 14:35:26.269428,0 days 00:00:01.035422,NaN,NaN,GradientBoosting,NaN,NaN,0.644460,10.0,9.0,6.0,157.0,NaN,COMPLETE
2,2,0.785847,2026-11-07 14:35:26.270591,2026-11-07 14:35:26.505940,0 days 00:00:00.235349,60.082923,NaN,SVM,auto,linear,NaN,NaN,NaN,NaN,NaN,NaN,COMPLETE
3,3,0.750466,2026-11-07 14:35:26.507027,2026-11-07 14:35:28.375214,0 days 00:00:01.868187,NaN,NaN,GradientBoosting,NaN,NaN,0.233228,6.0,1.0,5.0,186.0,NaN,COMPLETE
4,4,0.713222,2026-11-07 14:35:28.376311,2026-11-07 14:35:28.421539,0 days 00:00:00.045228,64.100012,NaN,SVM,scale,rbf,NaN,NaN,NaN,NaN,NaN,NaN,COMPLETE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,195,0.785847,2026-11-07 14:36:16.161411,2026-11-07 14:36:16.206337,0 days 00:00:00.044926,0.073318,NaN,SVM,scale,linear,NaN,NaN,NaN,NaN,NaN,"{""classifier"": ""SVM""}",COMPLETE
196,196,0.787709,2026-11-07 14:36:16.209537,2026-11-07 14:36:16.243083,0 days 00:00:00.033546,0.102761,NaN,SVM,scale,linear,NaN,NaN,NaN,NaN,NaN,"{""classifier"": ""SVM""}",COMPLETE
197,197,0.757914,2026-11-07 14:36:16.244102,2026-11-07 14:36:17.771783,0 days 00:00:01.527681,NaN,NaN,GradientBoosting,NaN,NaN,0.067972,8.0,9.0,10.0,163.0,"{""classifier"": ""GradientBoosting""}",COMPLETE
198,198,0.785847,2026-11-07 14:36:17.772889,2026-11-07 14:36:17.804551,0 days 00:00:00.031662,0.213190,NaN,SVM,scale,linear,NaN,NaN,NaN,NaN,NaN,"{""classifier"": ""SVM""}",COMPLETE


In [49]:
study.trials_dataframe()['params_classifier'].value_counts()

,count
params_classifier,
SVM,172
GradientBoosting,15
RandomForest,13
